In [1]:
# ==========================================================
# BYTE PAIR ENCODING (BPE) FROM SCRATCH USING PYTHON + NUMPY
# ==========================================================

import numpy as np
from collections import defaultdict

# ==========================================================
# Step 1: Create Training Corpus
# ==========================================================

training_corpus = np.array([
    "happy",
    "happier",
    "happiest",
    "play",
    "player",
    "playing",
    "talk",
    "talked",
    "talking",
    "learn",
    "learner",
    "learning"
])

print("="*60)
print("TRAINING CORPUS")
print("="*60)
print(training_corpus)

# ==========================================================
# Step 2: Preprocess Corpus
# ==========================================================

training_corpus = [word.lower().strip() for word in training_corpus]

# ==========================================================
# Step 3: Create Initial Vocabulary
# ==========================================================

token_dictionary = {}

for word in training_corpus:

    split_word = " ".join(list(word)) + " </w>"

    token_dictionary[split_word] = token_dictionary.get(split_word, 0) + 1

print("\n" + "="*60)
print("INITIAL VOCABULARY")
print("="*60)

for word, count in token_dictionary.items():
    print(f"{word}  -->  {count}")

# ==========================================================
# Step 4: Count Word Frequencies
# ==========================================================

frequency_array = np.array(list(token_dictionary.values()))

print("\nTotal Words :", np.sum(frequency_array))
print("Highest Frequency :", np.max(frequency_array))

# ==========================================================
# Step 5: Find Adjacent Symbol Pair Frequencies
# ==========================================================

def calculate_pair_frequency(vocabulary):

    pair_frequency = defaultdict(int)

    for word, count in vocabulary.items():

        symbols = word.split()

        for i in range(len(symbols)-1):

            pair = (symbols[i], symbols[i+1])

            pair_frequency[pair] += count

    return pair_frequency

# ==========================================================
# Step 6: Merge Two Symbols
# ==========================================================

def combine_symbols(best_pair, vocabulary):

    updated_vocab = {}

    old_token = " ".join(best_pair)
    new_token = "".join(best_pair)

    for word in vocabulary:

        merged = word.replace(old_token, new_token)

        updated_vocab[merged] = vocabulary[word]

    return updated_vocab

# ==========================================================
# Step 7: Learn Merge Rules
# ==========================================================

learned_merges = []

number_of_merges = 8

print("\n" + "="*60)
print("LEARNING MERGES")
print("="*60)

for step in range(number_of_merges):

    pair_frequency = calculate_pair_frequency(token_dictionary)

    if len(pair_frequency) == 0:
        break

    pair_list = list(pair_frequency.keys())

    pair_count = np.array(list(pair_frequency.values()))

    best_pair = pair_list[np.argmax(pair_count)]

    learned_merges.append(best_pair)

    print(f"\nMerge {step+1}")
    print("Selected Pair :", best_pair)
    print("Frequency     :", pair_frequency[best_pair])

    token_dictionary = combine_symbols(best_pair, token_dictionary)

# ==========================================================
# Step 8: Build Final Vocabulary
# ==========================================================

final_tokens = set()

for word in token_dictionary:

    for token in word.split():

        final_tokens.add(token)

print("\n" + "="*60)
print("FINAL VOCABULARY")
print("="*60)

print(sorted(final_tokens))

print("\nVocabulary Size :", len(final_tokens))

# ==========================================================
# Step 9: Encoder
# ==========================================================

def encode(input_word, merge_rules):

    token_list = list(input_word) + ["</w>"]

    for pair in merge_rules:

        i = 0

        while i < len(token_list)-1:

            if token_list[i] == pair[0] and token_list[i+1] == pair[1]:

                token_list = (
                    token_list[:i]
                    + ["".join(pair)]
                    + token_list[i+2:]
                )

            else:
                i += 1

    return token_list

# ==========================================================
# Step 10: Decoder
# ==========================================================

def decode(token_list):

    output = ""

    for token in token_list:

        if token != "</w>":
            output += token

    return output

# ==========================================================
# Step 11: Test the Tokenizer
# ==========================================================

test_words = np.array([
    "happiest",
    "playing",
    "learner",
    "talking"
])

print("\n" + "="*60)
print("ENCODING")
print("="*60)

for word in test_words:

    encoded = encode(word, learned_merges)

    print(f"{word:10} --> {encoded}")

print("\n" + "="*60)
print("DECODING")
print("="*60)

for word in test_words:

    encoded = encode(word, learned_merges)

    decoded = decode(encoded)

    print(f"{encoded} --> {decoded}")

# ==========================================================
# Step 12: Display Learned Merge Rules
# ==========================================================

print("\n" + "="*60)
print("LEARNED MERGE RULES")
print("="*60)

for i, rule in enumerate(learned_merges, start=1):

    print(f"{i}. {rule[0]} + {rule[1]}  -->  {rule[0]+rule[1]}")

print("\n" + "="*60)
print("BPE TOKENIZER COMPLETED SUCCESSFULLY")
print("="*60)

TRAINING CORPUS
['happy' 'happier' 'happiest' 'play' 'player' 'playing' 'talk' 'talked'
 'talking' 'learn' 'learner' 'learning']

INITIAL VOCABULARY
h a p p y </w>  -->  1
h a p p i e r </w>  -->  1
h a p p i e s t </w>  -->  1
p l a y </w>  -->  1
p l a y e r </w>  -->  1
p l a y i n g </w>  -->  1
t a l k </w>  -->  1
t a l k e d </w>  -->  1
t a l k i n g </w>  -->  1
l e a r n </w>  -->  1
l e a r n e r </w>  -->  1
l e a r n i n g </w>  -->  1

Total Words : 12
Highest Frequency : 1

LEARNING MERGES

Merge 1
Selected Pair : ('h', 'a')
Frequency     : 3

Merge 2
Selected Pair : ('ha', 'p')
Frequency     : 3

Merge 3
Selected Pair : ('hap', 'p')
Frequency     : 3

Merge 4
Selected Pair : ('e', 'r')
Frequency     : 3

Merge 5
Selected Pair : ('er', '</w>')
Frequency     : 3

Merge 6
Selected Pair : ('p', 'l')
Frequency     : 3

Merge 7
Selected Pair : ('pl', 'a')
Frequency     : 3

Merge 8
Selected Pair : ('pla', 'y')
Frequency     : 3

FINAL VOCABULARY
['</w>', 'a', 'd', 'e', 'er</w